# Logic Lab — Logical Reasoning for Planning

**Course:** CS F407 — Artificial Intelligence · **Lab:** Logical Planning
**Handout:** `logic_lab_ex.pdf`

---

The central claim of this lab is

$$\textbf{Logic} + \textbf{Search} = \textbf{Planning}$$

and the notebook is organised to make each half visible separately. **Logic**
decides whether $S \models \text{Preconditions}(a)$ — what is *possible*.
**Search** decides which applicable sequences to try — what to *do*.

Two things go beyond the literal instructions, because they turn the handout's
warnings into results:

1. **Task 3 Test C warns against confusing "the robot reached C" with "the
   package reached C".** Rather than only checking that the planner avoids this,
   the notebook builds a *deliberately broken* planner that drops the
   precondition check, runs it on the same problem, and shows the invalid plan it
   produces. The failure is exhibited, not described.
2. **Task 5 asks whether to trust an LLM's explanation or the executed state
   transitions.** The notebook implements an independent `verify_plan` that
   replays a plan against the action definitions, and then runs it on a plan that
   is *wrong but plausible* — the kind an explanation would happily justify.

In [1]:
"""A small STRIPS-style planner: states are sets of ground propositions."""
import itertools
from collections import deque
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Action:
    """A STRIPS action.

    Negative preconditions are included because the handout's specification asks
    for them; the warehouse domain happens not to need any, which is itself worth
    noting when reading the generated code.
    """
    name: str
    pre_pos: frozenset = frozenset()   # must be present in the state
    pre_neg: frozenset = frozenset()   # must be absent from the state
    eff_add: frozenset = frozenset()   # becomes true
    eff_del: frozenset = frozenset()   # becomes false

    def applicable(self, state):
        """S |= Preconditions(a).  This is the LOGICAL half of the planner."""
        return self.pre_pos <= state and not (self.pre_neg & state)

    def apply(self, state):
        """Apply(S, a): delete first, then add -- the order matters when a
        proposition appears in both effect lists."""
        if not self.applicable(state):
            raise ValueError(f"{self.name} is not applicable in {sorted(state)}")
        return frozenset((state - self.eff_del) | self.eff_add)

    def __str__(self):
        return self.name


def at(thing, place):
    return f"At({thing},{place})"

HOLDING = "Holding(Package)"
LOCATIONS = ["A", "B", "C"]
CONNECTED = [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]


def build_actions(include_pickup=True, include_drop=True, include_move=True):
    """The warehouse action set, with parts omittable for the Task 3 tests."""
    actions = []
    if include_move:
        for x, y in CONNECTED:
            actions.append(Action(
                name=f"Move({x},{y})",
                pre_pos=frozenset({at("Robot", x)}),
                eff_add=frozenset({at("Robot", y)}),
                eff_del=frozenset({at("Robot", x)})))
    if include_pickup:
        for loc in LOCATIONS:
            actions.append(Action(
                name=f"PickUp(Package,{loc})",
                pre_pos=frozenset({at("Robot", loc), at("Package", loc)}),
                eff_add=frozenset({HOLDING}),
                eff_del=frozenset({at("Package", loc)})))
    if include_drop:
        for loc in LOCATIONS:
            actions.append(Action(
                name=f"Drop(Package,{loc})",
                pre_pos=frozenset({at("Robot", loc), HOLDING}),
                eff_add=frozenset({at("Package", loc)}),
                eff_del=frozenset({HOLDING})))
    return actions


INITIAL = frozenset({at("Robot", "A"), at("Package", "A")})
GOAL = frozenset({at("Package", "C")})
ACTIONS = build_actions()

print(f"Initial state I = {{{', '.join(sorted(INITIAL))}}}")
print(f"Goal        G = {{{', '.join(sorted(GOAL))}}}")
print(f"\n{len(ACTIONS)} ground actions:")
for a in ACTIONS:
    print(f"  {a.name:24s} pre+: {sorted(a.pre_pos)}")
    print(f"  {'':24s} add : {sorted(a.eff_add)}   del: {sorted(a.eff_del)}")

Initial state I = {At(Package,A), At(Robot,A)}
Goal        G = {At(Package,C)}

10 ground actions:
  Move(A,B)                pre+: ['At(Robot,A)']
                           add : ['At(Robot,B)']   del: ['At(Robot,A)']
  Move(B,A)                pre+: ['At(Robot,B)']
                           add : ['At(Robot,A)']   del: ['At(Robot,B)']
  Move(B,C)                pre+: ['At(Robot,B)']
                           add : ['At(Robot,C)']   del: ['At(Robot,B)']
  Move(C,B)                pre+: ['At(Robot,C)']
                           add : ['At(Robot,B)']   del: ['At(Robot,C)']
  PickUp(Package,A)        pre+: ['At(Package,A)', 'At(Robot,A)']
                           add : ['Holding(Package)']   del: ['At(Package,A)']
  PickUp(Package,B)        pre+: ['At(Package,B)', 'At(Robot,B)']
                           add : ['Holding(Package)']   del: ['At(Package,B)']
  PickUp(Package,C)        pre+: ['At(Package,C)', 'At(Robot,C)']
                           add : ['Holding(Package)']   del: 

---

## Task 0 — Understanding the planning problem

**(a) Initial state.** $I = \{\text{At}(Robot, A),\ \text{At}(Package, A)\}$.

**(b) Goal.** $G = \{\text{At}(Package, C)\}$. The goal is a *partial* state — it
constrains only where the package is. Where the robot ends up is unconstrained,
and any state containing $\text{At}(Package,C)$ satisfies the goal. This is why
the goal test is $G \subseteq S$ rather than $G = S$.

**(c) and (d) Actions, preconditions, effects.**

| Action | Preconditions | Add | Delete |
|---|---|---|---|
| $\text{Move}(x,y)$ | $\text{At}(Robot,x)$ | $\text{At}(Robot,y)$ | $\text{At}(Robot,x)$ |
| $\text{PickUp}(P,\ell)$ | $\text{At}(Robot,\ell) \wedge \text{At}(P,\ell)$ | $\text{Holding}(P)$ | $\text{At}(P,\ell)$ |
| $\text{Drop}(P,\ell)$ | $\text{At}(Robot,\ell) \wedge \text{Holding}(P)$ | $\text{At}(P,\ell)$ | $\text{Holding}(P)$ |

Move is available only for *connected* pairs: A–B, B–A, B–C, C–B. There is no
direct A–C link, which is the fact the Prolog verifier in Task 7 checks.

### Which actions are applicable in $I$?

**Is $\text{PickUp}(Package, A)$ applicable?** Its preconditions are
$\text{At}(Robot,A)$ and $\text{At}(Package,A)$. Both are in $I$, so
$I \models \text{Preconditions}$ — **yes**.

**Is $\text{Drop}(Package, C)$ applicable?** Its preconditions are
$\text{At}(Robot,C)$ and $\text{Holding}(Package)$. Neither is in $I$ — the robot
is at A and holds nothing — so **no**. It is in the action *list*, but being
listed is not the same as being applicable. That distinction is exactly where
logical reasoning enters planning.

In [2]:
def applicable_actions(state, actions):
    return [a for a in actions if a.applicable(state)]

print("Applicability check in the initial state I:\n")
print("| action | preconditions | satisfied by I? | applicable |")
print("|---|---|---|:---:|")
for a in ACTIONS:
    missing = sorted(a.pre_pos - INITIAL)
    sat = "all present" if not missing else f"missing {missing}"
    print(f"| {a.name} | {sorted(a.pre_pos)} | {sat} | "
          f"{'YES' if a.applicable(INITIAL) else 'no'} |")

app = applicable_actions(INITIAL, ACTIONS)
print(f"\nApplicable in I: {[a.name for a in app]}")

pickup_a = next(a for a in ACTIONS if a.name == "PickUp(Package,A)")
drop_c   = next(a for a in ACTIONS if a.name == "Drop(Package,C)")
print(f"\nPickUp(Package,A) applicable in I : {pickup_a.applicable(INITIAL)}")
print(f"Drop(Package,C)   applicable in I : {drop_c.applicable(INITIAL)}")
print(f"  -> Drop is blocked because I lacks {sorted(drop_c.pre_pos - INITIAL)}")
assert pickup_a.applicable(INITIAL)
assert not drop_c.applicable(INITIAL)

Applicability check in the initial state I:

| action | preconditions | satisfied by I? | applicable |
|---|---|---|:---:|
| Move(A,B) | ['At(Robot,A)'] | all present | YES |
| Move(B,A) | ['At(Robot,B)'] | missing ['At(Robot,B)'] | no |
| Move(B,C) | ['At(Robot,B)'] | missing ['At(Robot,B)'] | no |
| Move(C,B) | ['At(Robot,C)'] | missing ['At(Robot,C)'] | no |
| PickUp(Package,A) | ['At(Package,A)', 'At(Robot,A)'] | all present | YES |
| PickUp(Package,B) | ['At(Package,B)', 'At(Robot,B)'] | missing ['At(Package,B)', 'At(Robot,B)'] | no |
| PickUp(Package,C) | ['At(Package,C)', 'At(Robot,C)'] | missing ['At(Package,C)', 'At(Robot,C)'] | no |
| Drop(Package,A) | ['At(Robot,A)', 'Holding(Package)'] | missing ['Holding(Package)'] | no |
| Drop(Package,B) | ['At(Robot,B)', 'Holding(Package)'] | missing ['At(Robot,B)', 'Holding(Package)'] | no |
| Drop(Package,C) | ['At(Robot,C)', 'Holding(Package)'] | missing ['At(Robot,C)', 'Holding(Package)'] | no |

Applicable in I: ['Move(A,B)', 'Pick

---

## Task 1 — Constructing a plan by hand

Before any code, the plan reasoned out on paper. The package starts at A with the
robot, so there is no need to move before picking up:

$$\text{PickUp}(P, A),\ \text{Move}(A,B),\ \text{Move}(B,C),\ \text{Drop}(P,C)$$

| State | Facts | Justification |
|---|---|---|
| $S_0$ | At(Robot,A), At(Package,A) | the given $I$ |
| $S_1$ | At(Robot,A), Holding(P) | PickUp(P,A): both preconditions hold in $S_0$; deletes At(P,A), adds Holding(P) |
| $S_2$ | At(Robot,B), Holding(P) | Move(A,B): At(Robot,A) holds in $S_1$; deletes it, adds At(Robot,B) |
| $S_3$ | At(Robot,C), Holding(P) | Move(B,C): At(Robot,B) holds in $S_2$ |
| $S_4$ | At(Robot,C), At(Package,C) | Drop(P,C): At(Robot,C) and Holding(P) both hold in $S_3$ |

$S_4 \supseteq G$, so the plan achieves the goal in **4 actions**.

Note that the handout's illustrative sequence — Move(A,B), **PickUp(P,B)**,
Move(B,C), Drop(P,C) — is *not* valid: after Move(A,B) the package is still at A,
so PickUp(P,B) fails its precondition At(Package,B). This is precisely the trap
the lab is about, and the notebook checks it below rather than taking my word
for it.

### Is 4 actions the minimum?

Any valid plan must include a PickUp and a Drop (nothing else changes the
package's location), and the robot must travel from A to C, which needs at least
two Moves because there is no A–C connection. So $\ge 4$ actions, and the plan
above achieves it.

In [3]:
def verify_plan(initial, goal, plan, label="plan"):
    """Independently replay a plan against the action definitions.

    This does NOT use the planner. It is the check that makes a generated plan
    -- or a generated *explanation* -- trustworthy, by recomputing every state
    transition and every precondition test from scratch.
    """
    state = frozenset(initial)
    trace = [("(initial)", state, True, [])]
    valid = True
    for action in plan:
        missing = sorted(action.pre_pos - state)
        blocked = sorted(action.pre_neg & state)
        ok = not missing and not blocked
        if not ok:
            valid = False
            trace.append((action.name, state, False, missing + [f"!{b}" for b in blocked]))
            break
        state = action.apply(state)
        trace.append((action.name, state, True, []))
    achieved = valid and goal <= state
    return {"label": label, "valid": valid, "achieves_goal": achieved,
            "final_state": state, "trace": trace}


def show_verification(report):
    print(f"--- {report['label']} ---")
    for name, state, ok, missing in report["trace"]:
        facts = "{" + ", ".join(sorted(state)) + "}"
        if ok:
            print(f"  {name:24s} -> {facts}")
        else:
            print(f"  {name:24s} -> BLOCKED, preconditions not satisfied: {missing}")
    print(f"  every action applicable : {report['valid']}")
    print(f"  goal achieved           : {report['achieves_goal']}\n")


by_name = {a.name: a for a in ACTIONS}
hand_plan = [by_name["PickUp(Package,A)"], by_name["Move(A,B)"],
             by_name["Move(B,C)"], by_name["Drop(Package,C)"]]
show_verification(verify_plan(INITIAL, GOAL, hand_plan, "my hand-built plan"))

# The handout's illustrative sequence, checked rather than assumed
handout_plan = [by_name["Move(A,B)"], by_name["PickUp(Package,B)"],
                by_name["Move(B,C)"], by_name["Drop(Package,C)"]]
show_verification(verify_plan(INITIAL, GOAL, handout_plan,
                              "the handout's illustrative sequence"))
print("The second sequence LOOKS reasonable -- it has the right shape, the right")
print("number of steps, and the right actions in a sensible order. It is still")
print("invalid, because after Move(A,B) the package has not moved and")
print("PickUp(Package,B) fails its precondition At(Package,B).")
print("This is the whole point of Reflection Q3: plausible is not valid.")

--- my hand-built plan ---
  (initial)                -> {At(Package,A), At(Robot,A)}
  PickUp(Package,A)        -> {At(Robot,A), Holding(Package)}
  Move(A,B)                -> {At(Robot,B), Holding(Package)}
  Move(B,C)                -> {At(Robot,C), Holding(Package)}
  Drop(Package,C)          -> {At(Package,C), At(Robot,C)}
  every action applicable : True
  goal achieved           : True

--- the handout's illustrative sequence ---
  (initial)                -> {At(Package,A), At(Robot,A)}
  Move(A,B)                -> {At(Package,A), At(Robot,B)}
  PickUp(Package,B)        -> BLOCKED, preconditions not satisfied: ['At(Package,B)']
  every action applicable : False
  goal achieved           : False

The second sequence LOOKS reasonable -- it has the right shape, the right
number of steps, and the right actions in a sensible order. It is still
invalid, because after Move(A,B) the package has not moved and
PickUp(Package,B) fails its precondition At(Package,B).
This is the whole po

---

## Task 2 — Asking an LLM to implement the planner

### The prompt I used

> Implement a small **STRIPS-style planner** in Python. Follow this
> specification exactly; do not substitute a library or a different formalism.
>
> **State.** A `frozenset` of ground proposition strings. Immutable and hashable,
> because states will be used as dictionary keys in the search's visited set.
>
> **Action.** A frozen dataclass with `name`, `pre_pos`, `pre_neg`, `eff_add`,
> `eff_del`, all `frozenset`s, plus:
> - `applicable(state)` returning `pre_pos <= state and not (pre_neg & state)`;
> - `apply(state)` returning `(state - eff_del) | eff_add` — **delete before
>   add**, and it must raise rather than silently proceed if the action is not
>   applicable.
>
> **Planner.** Breadth-first search over states. The goal test is
> `goal <= state` (subset), **not** equality, because the goal is partial.
> Keep a visited set of states so the search terminates. Return the action
> sequence *and* the state after each action; return an explicit "no plan"
> result when the frontier empties, rather than raising or looping.
>
> **Required details**, because these are what I want to check:
> - the visited set must hold **states**, not actions — a state reached by two
>   different routes is the same search node;
> - `apply` must delete before adding, so an action whose add and delete lists
>   overlap behaves correctly;
> - report the number of states expanded, so BFS's cost is visible;
> - the goal test must be a subset test; using equality would make the goal
>   unreachable here, since the final state also records where the robot is.
>
> Also write a **separate** `verify_plan(initial, goal, plan)` that replays a
> plan against the action definitions without using the planner, reporting for
> each step whether preconditions held and what the resulting state was. I want
> to be able to check a plan I did not generate.

### Where the specification's ideas appear in the code

| Idea | Where |
|---|---|
| **Preconditions** → when is an action applicable? | `Action.applicable` — the subset/disjointness test $S \models \text{Pre}(a)$ |
| **Effects** → how does the state change? | `Action.apply` — `(state - eff_del) | eff_add` |
| **Goal** → when does planning terminate? | `goal <= state` in the planner's loop |
| **BFS** → how are alternatives explored? | the `deque` frontier, expanding shallowest-first |

### Corrections I made to the generated code

1. **The goal test was `state == goal`.** On this problem that is unsatisfiable:
   the final state is $\{\text{At}(Robot,C), \text{At}(Package,C)\}$, which is a
   proper superset of $G$. The planner reported "no plan found" for a problem
   that plainly has one — a loud failure, and the easiest of the three to catch.
2. **The visited set stored *plans*, not states.** Every distinct action sequence
   is a distinct plan, so nothing was ever pruned and the search explored an
   exponentially growing tree. It still terminated here because the state space
   is tiny, but the expansion count was wildly inflated and it would not scale.
3. **`apply` added before deleting.** No warehouse action has overlapping add and
   delete lists, so this made no difference to any output — and that is exactly
   why it was worth fixing. It is a latent bug that would surface silently the
   first time the domain grew an action like "recharge while staying put".

In [4]:
@dataclass
class PlanResult:
    found: bool
    plan: list = field(default_factory=list)
    states: list = field(default_factory=list)   # state after each action
    n_expanded: int = 0
    n_generated: int = 0
    note: str = ""

    @property
    def length(self):
        return len(self.plan)


def bfs_planner(initial, goal, actions, max_expansions=100_000):
    """Logic + Search.

    SEARCH: breadth-first over states, so the first plan found is shortest.
    LOGIC : `action.applicable(state)` decides which successors even exist.
    """
    initial = frozenset(initial)
    goal = frozenset(goal)
    if goal <= initial:                       # subset test, not equality
        return PlanResult(True, [], [initial], 0, 0, "goal already true in I")

    frontier = deque([(initial, [])])
    visited = {initial}                       # visited holds STATES
    n_expanded = n_generated = 0

    while frontier:
        state, plan = frontier.popleft()
        n_expanded += 1
        if n_expanded > max_expansions:
            return PlanResult(False, note="expansion limit reached")
        for action in actions:
            if not action.applicable(state):  # <-- the logical filter
                continue
            n_generated += 1
            nxt = action.apply(state)
            if nxt in visited:
                continue
            new_plan = plan + [action]
            if goal <= nxt:
                states, s = [], initial
                for a in new_plan:
                    s = a.apply(s)
                    states.append(s)
                return PlanResult(True, new_plan, states, n_expanded, n_generated)
            visited.add(nxt)
            frontier.append((nxt, new_plan))

    return PlanResult(False, n_expanded=n_expanded, n_generated=n_generated,
                      note="frontier exhausted: no plan exists")


def show_plan(result, label=""):
    if label:
        print(f"--- {label} ---")
    if not result.found:
        print(f"  No plan found. ({result.note})")
        print(f"  states expanded: {result.n_expanded}\n")
        return
    print(f"  Plan of length {result.length}, found after expanding "
          f"{result.n_expanded} states:\n")
    print(f"  {'(initial)':26s} {{{', '.join(sorted(INITIAL))}}}")
    for i, (action, state) in enumerate(zip(result.plan, result.states), 1):
        print(f"  {i}. {action.name:23s} {{{', '.join(sorted(state))}}}")
    print()


result = bfs_planner(INITIAL, GOAL, ACTIONS)
show_plan(result, "Task 3 Test A: the original warehouse problem")

report = verify_plan(INITIAL, GOAL, result.plan, "independent verification")
print(f"Independent replay: every action applicable = {report['valid']}, "
      f"goal achieved = {report['achieves_goal']}")
assert result.found and report["achieves_goal"]
assert result.length == 4, "4 actions is the proven minimum"
print(f"Plan length {result.length} matches the hand-derived minimum of 4.")

--- Task 3 Test A: the original warehouse problem ---
  Plan of length 4, found after expanding 6 states:

  (initial)                  {At(Package,A), At(Robot,A)}
  1. PickUp(Package,A)       {At(Robot,A), Holding(Package)}
  2. Move(A,B)               {At(Robot,B), Holding(Package)}
  3. Move(B,C)               {At(Robot,C), Holding(Package)}
  4. Drop(Package,C)         {At(Package,C), At(Robot,C)}

Independent replay: every action applicable = True, goal achieved = True
Plan length 4 matches the hand-derived minimum of 4.


---

## Task 3 — Testing the generated planner

### Test B — an impossible problem

Removing `PickUp` should make the goal unreachable, and the planner must say so
rather than invent an action.

In [5]:
no_pickup = build_actions(include_pickup=False)
res_b = bfs_planner(INITIAL, GOAL, no_pickup)
show_plan(res_b, "Test B: PickUp removed")
print(f"  Reported no plan : {not res_b.found}")
print(f"  Reason           : {res_b.note}")
assert not res_b.found
print("\n  The planner exhausted a finite state space and stopped. It did not")
print("  hallucinate a Move that transports the package, and it did not hang.")

# A second impossible variant: the package is unreachable because C is disconnected
disconnected = [a for a in ACTIONS if a.name not in ("Move(B,C)", "Move(C,B)")]
res_b2 = bfs_planner(INITIAL, GOAL, disconnected)
print(f"\n  Variant: remove the B-C link  -> plan found: {res_b2.found} "
      f"({res_b2.note})")
assert not res_b2.found

--- Test B: PickUp removed ---
  No plan found. (frontier exhausted: no plan exists)
  states expanded: 3

  Reported no plan : True
  Reason           : frontier exhausted: no plan exists

  The planner exhausted a finite state space and stopped. It did not
  hallucinate a Move that transports the package, and it did not hang.

  Variant: remove the B-C link  -> plan found: False (frontier exhausted: no plan exists)


### Test C — irrelevant actions, and the trap the handout warns about

The handout's warning is that the planner must not treat *the robot* reaching C
as equivalent to *the package* reaching C. The propositions are distinct —
$\text{At}(Robot,C)$ and $\text{At}(Package,C)$ — so a correct planner cannot
confuse them.

Rather than only assert that, the cell below adds genuinely irrelevant actions
(the robot pacing between A and B while the package sits still) and confirms the
planner ignores them. Then it does something stronger: it **breaks the planner on
purpose** by removing the precondition check, and shows the invalid plan that
results.

In [6]:
# 1. Irrelevant actions present: does the planner still find the 4-step plan?
idle = Action(name="Idle", pre_pos=frozenset(), eff_add=frozenset(),
              eff_del=frozenset())
with_noise = ACTIONS + [idle]
res_c = bfs_planner(INITIAL, GOAL, with_noise)
print(f"With an Idle action added: plan length {res_c.length}, "
      f"plan = {[a.name for a in res_c.plan]}")
assert res_c.length == 4
print("  -> BFS still returns a shortest plan; the useless action is simply never")
print("     part of one.\n")

# 2. Does reaching C with the robot satisfy the goal? It must not.
robot_at_c = frozenset({at("Robot", "C"), at("Package", "A")})
print(f"State with the ROBOT at C but the package at A: "
      f"{{{', '.join(sorted(robot_at_c))}}}")
print(f"  goal {sorted(GOAL)} satisfied? {GOAL <= robot_at_c}")
assert not (GOAL <= robot_at_c)
print("  -> correct: At(Robot,C) and At(Package,C) are different propositions.")

With an Idle action added: plan length 4, plan = ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)']
  -> BFS still returns a shortest plan; the useless action is simply never
     part of one.

State with the ROBOT at C but the package at A: {At(Package,A), At(Robot,C)}
  goal ['At(Package,C)'] satisfied? False
  -> correct: At(Robot,C) and At(Package,C) are different propositions.


In [7]:
# 3. The interesting test: a planner that SKIPS the precondition check.
def broken_planner(initial, goal, actions, max_expansions=20_000):
    """Deliberately wrong: applies actions without testing S |= Pre(a).

    This is the single most likely thing to go missing in generated code,
    because the program still runs and still returns a plan.
    """
    initial, goal = frozenset(initial), frozenset(goal)
    frontier = deque([(initial, [])])
    visited = {initial}
    n = 0
    while frontier and n < max_expansions:
        state, plan = frontier.popleft()
        n += 1
        for action in actions:
            # NOTE: no applicability test -- effects applied unconditionally
            nxt = frozenset((state - action.eff_del) | action.eff_add)
            if nxt in visited:
                continue
            new_plan = plan + [action]
            if goal <= nxt:
                return PlanResult(True, new_plan, [], n, 0, "from the BROKEN planner")
            visited.add(nxt)
            frontier.append((nxt, new_plan))
    return PlanResult(False, note="no plan")

bad = broken_planner(INITIAL, GOAL, ACTIONS)
print(f"The broken planner returns a plan of length {bad.length}:")
print(f"  {[a.name for a in bad.plan]}\n")

bad_report = verify_plan(INITIAL, GOAL, bad.plan, "verifying the broken planner's plan")
show_verification(bad_report)
assert not bad_report["achieves_goal"], "the broken plan should fail verification"

print("This is the failure the handout warns about, made concrete. The broken")
print("planner produced a SHORTER plan than the correct one, which is exactly")
print("what makes it dangerous: it looks better. Dropping the precondition test")
print("lets the robot drop a package it was never holding, or teleport between")
print("unconnected rooms -- the search space grows, so shorter 'solutions' appear,")
print("and every one of them is fiction.")
print("\nNote that the broken planner RAN FINE. No exception, no hang, plausible")
print("output. Only independent verification exposes it.")

The broken planner returns a plan of length 1:
  ['Drop(Package,C)']

--- verifying the broken planner's plan ---
  (initial)                -> {At(Package,A), At(Robot,A)}
  Drop(Package,C)          -> BLOCKED, preconditions not satisfied: ['At(Robot,C)', 'Holding(Package)']
  every action applicable : False
  goal achieved           : False

This is the failure the handout warns about, made concrete. The broken
planner produced a SHORTER plan than the correct one, which is exactly
what makes it dangerous: it looks better. Dropping the precondition test
lets the robot drop a package it was never holding, or teleport between
unconnected rooms -- the search space grows, so shorter 'solutions' appear,
and every one of them is fiction.

Note that the broken planner RAN FINE. No exception, no hang, plausible
output. Only independent verification exposes it.


In [8]:
print("Summary of Task 3 tests:\n")
print("| test | initial state | goal | plan found | plan | valid? |")
print("|---|---|---|:---:|---|:---:|")

rows = [
    ("A: original", INITIAL, GOAL, ACTIONS),
    ("B: PickUp removed", INITIAL, GOAL, no_pickup),
    ("B2: B-C link removed", INITIAL, GOAL, disconnected),
    ("C: with Idle action", INITIAL, GOAL, with_noise),
]
for label, init, goal, acts in rows:
    r = bfs_planner(init, goal, acts)
    if r.found:
        v = verify_plan(init, goal, r.plan)
        plan_str = " -> ".join(a.name for a in r.plan)
        print(f"| {label} | At(Robot,A), At(Package,A) | At(Package,C) | yes | "
              f"{plan_str} | {'yes' if v['achieves_goal'] else 'NO'} |")
    else:
        print(f"| {label} | At(Robot,A), At(Package,A) | At(Package,C) | **no** | - | n/a |")

print(f"| C2: broken planner | At(Robot,A), At(Package,A) | At(Package,C) | yes | "
      f"{' -> '.join(a.name for a in bad.plan)} | **NO** |")
print("\nEvery plan the CORRECT planner returned passes independent verification.")
print("The one plan that fails verification came from the planner missing its")
print("logical component.")

Summary of Task 3 tests:

| test | initial state | goal | plan found | plan | valid? |
|---|---|---|:---:|---|:---:|
| A: original | At(Robot,A), At(Package,A) | At(Package,C) | yes | PickUp(Package,A) -> Move(A,B) -> Move(B,C) -> Drop(Package,C) | yes |
| B: PickUp removed | At(Robot,A), At(Package,A) | At(Package,C) | **no** | - | n/a |
| B2: B-C link removed | At(Robot,A), At(Package,A) | At(Package,C) | **no** | - | n/a |
| C: with Idle action | At(Robot,A), At(Package,A) | At(Package,C) | yes | PickUp(Package,A) -> Move(A,B) -> Move(B,C) -> Drop(Package,C) | yes |
| C2: broken planner | At(Robot,A), At(Package,A) | At(Package,C) | yes | Drop(Package,C) | **NO** |

Every plan the CORRECT planner returned passes independent verification.
The one plan that fails verification came from the planner missing its
logical component.


---

## Task 4 — Logic and search

### Completing the handout's diagram

```
                     Current state S
                            |
                            v
              Check action preconditions      <-- LOGIC
              does  S |= Pre(a)  hold?
                            |
              +-------------+-------------+
              | no                        | yes
              v                           v
        discard a                Apply the effects       <-- LOGIC
      (never generated)          S' = (S \ Del) u Add
                                          |
                                          v
                              Generate successor state S'
                                          |
                                          v
                          Search over alternatives        <-- SEARCH
                        (BFS: enqueue S', expand shallowest first,
                         skip states already visited)
                                          |
                                          v
                                    Goal?  G subset of S'
                                     |          |
                                  no |          | yes
                                     v          v
                              keep searching   return the plan
```

The missing box the handout leaves blank is **"apply the effects"** — the second
half of the logical component. Logic does two distinct jobs here, and it is worth
separating them: it decides *whether* an action may be taken (the precondition
entailment test) and it decides *what the world then becomes* (the
delete-then-add update). Search does neither; it only chooses which of the
logically-permitted successors to explore next.

### In my own words

The two components answer different questions and neither can do the other's job.

**Logic determines what is possible.** Given a state $S$ and an action $a$, the
test $S \models \text{Pre}(a)$ is a purely logical question about entailment —
are the required propositions present, and are the forbidden ones absent? It
involves no lookahead and no notion of the goal. It is implemented as one line,
`pre_pos <= state and not (pre_neg & state)`, and it is the *only* thing standing
between the planner and nonsense: Test C showed that removing it lets the robot
drop a package it never picked up.

**Search determines what to try.** Even with every action correctly filtered,
there are many applicable sequences, and most lead nowhere. Search imposes an
order on exploring them — here BFS, which expands shallowest-first and therefore
returns a shortest plan. Search knows nothing about warehouses; swap the domain
and the same BFS works unchanged.

The interface between them is narrow and that is the point: logic exposes a
successor function, search consumes it. This is also exactly the relationship
from the Search lab — the planner *is* a search problem, with states as sets of
propositions rather than grid coordinates, and the successor function computed by
logical inference rather than by reading a map.

### Why the state space is finite

Worth noting, because it is what makes "no plan found" a *proof* rather than a
timeout. The robot is at one of 3 locations; the package is at one of 3 locations
or held. So at most $3 \times 4 = 12$ reachable states. BFS with a visited set
must therefore terminate, and when the frontier empties, no plan exists.

In [9]:
# The state space really is small and closed -- enumerate it.
def reachable_states(initial, actions):
    seen = {frozenset(initial)}
    q = deque([frozenset(initial)])
    edges = 0
    while q:
        s = q.popleft()
        for a in actions:
            if a.applicable(s):
                edges += 1
                n = a.apply(s)
                if n not in seen:
                    seen.add(n); q.append(n)
    return seen, edges

states, edges = reachable_states(INITIAL, ACTIONS)
print(f"Reachable states from I: {len(states)} (theoretical bound 3 x 4 = 12)")
print(f"Applicable-action edges : {edges}\n")
for s in sorted(states, key=lambda x: sorted(x)):
    mark = "  <- goal" if GOAL <= s else ""
    print(f"  {{{', '.join(sorted(s))}}}{mark}")
print(f"\nBecause the space is finite and BFS marks visited states, an empty")
print("frontier PROVES no plan exists. Test B's 'no plan found' is a result,")
print("not a timeout.")

Reachable states from I: 12 (theoretical bound 3 x 4 = 12)
Applicable-action edges : 22

  {At(Package,A), At(Robot,A)}
  {At(Package,A), At(Robot,B)}
  {At(Package,A), At(Robot,C)}
  {At(Package,B), At(Robot,A)}
  {At(Package,B), At(Robot,B)}
  {At(Package,B), At(Robot,C)}
  {At(Package,C), At(Robot,A)}  <- goal
  {At(Package,C), At(Robot,B)}  <- goal
  {At(Package,C), At(Robot,C)}  <- goal
  {At(Robot,A), Holding(Package)}
  {At(Robot,B), Holding(Package)}
  {At(Robot,C), Holding(Package)}

Because the space is finite and BFS marks visited states, an empty
frontier PROVES no plan exists. Test B's 'no plan found' is a result,
not a timeout.


---

## Task 5 — Can the LLM verify its own plan?

I asked the LLM the question the handout suggests: *for every action in the plan,
identify its preconditions and show that those preconditions are satisfied in the
state in which the action is executed.*

It produced a fluent, correctly-formatted walkthrough of the four steps with a
state table after each action — and for the **correct** plan the walkthrough was
right. That is not the interesting case. The interesting case is what an
explanation does with a plan that is *wrong but plausible*, so I gave it the
handout's own illustrative sequence — Move(A,B), PickUp(P,B), Move(B,C),
Drop(P,C) — which is invalid.

### Which should you trust: (a) the explanation, or (b) the executed transitions?

**(b), and not as a matter of degree.** The two are not two opinions about the
same thing; they are different kinds of object.

An explanation is *generated text about* a computation. It is produced by the
same process, with the same failure modes, that produced the plan — so if the
plan is wrong because a precondition was overlooked, the explanation is generated
under that same misapprehension and will confidently narrate the missing step.
The fluency of the narration carries no information about its correctness.

Executed transitions are *the computation itself*. `verify_plan` re-derives each
state from the action definitions and tests each precondition as set containment.
It has no opinion and no way to be persuaded; if a precondition is absent, the
subset test is `False`. Crucially, it is **independent** of the planner — it
shares no code with it — so a bug in the planner cannot hide a bug in the
verification.

The general principle: **a generated explanation is not an independent
verification.** It is a second sample from the same distribution. The cell below
makes this concrete.

In [10]:
print("A plan that is WRONG but that an explanation would happily justify:\n")
show_verification(verify_plan(INITIAL, GOAL, handout_plan,
                              "Move(A,B), PickUp(P,B), Move(B,C), Drop(P,C)"))

print("An explanation of this plan writes itself, and every sentence sounds right:")
print('  "Move(A,B) takes the robot to B. PickUp(Package,B) picks up the package')
print('   at B. Move(B,C) carries it to C. Drop(Package,C) releases it, achieving')
print('   At(Package,C)."')
print()
print("Every clause is locally plausible and the conclusion is exactly the goal.")
print("The flaw is a single unstated assumption -- that the package moved with")
print("the robot -- which no amount of fluency exposes. The verifier finds it")
print("immediately, because At(Package,B) is simply not in the state.")
print()
print("Note the asymmetry: the explanation must be READ carefully to find the")
print("error; the verifier reports it as a boolean. Only one of those scales.")

A plan that is WRONG but that an explanation would happily justify:

--- Move(A,B), PickUp(P,B), Move(B,C), Drop(P,C) ---
  (initial)                -> {At(Package,A), At(Robot,A)}
  Move(A,B)                -> {At(Package,A), At(Robot,B)}
  PickUp(Package,B)        -> BLOCKED, preconditions not satisfied: ['At(Package,B)']
  every action applicable : False
  goal achieved           : False

An explanation of this plan writes itself, and every sentence sounds right:
  "Move(A,B) takes the robot to B. PickUp(Package,B) picks up the package
   at B. Move(B,C) carries it to C. Drop(Package,C) releases it, achieving
   At(Package,C)."

Every clause is locally plausible and the conclusion is exactly the goal.
The flaw is a single unstated assumption -- that the package moved with
the robot -- which no amount of fluency exposes. The verifier finds it
immediately, because At(Package,B) is simply not in the state.

Note the asymmetry: the explanation must be READ carefully to find the
error; t

---

# Optional Extension — Prolog as a Logical Verifier

The architecture the handout is pointing at:

$$\textbf{Generate} \;\longrightarrow\; \textbf{Independent verification}$$

Python generates a candidate plan. Prolog — a *separate* system, with its own
knowledge base and its own inference engine, sharing no code with the planner —
checks whether the proposed moves are supported by the logical description of the
warehouse. If both agree, the agreement means something, precisely because
neither could have caused the other's answer.

In [11]:
import shutil
import subprocess
import textwrap
from pathlib import Path

PROLOG = shutil.which("swipl")
print("SWI-Prolog found at:", PROLOG if PROLOG else "NOT FOUND")

PLANNER_PL = textwrap.dedent("""\
    % planner.pl -- warehouse knowledge base and an independent move verifier.

    % ---- Task 6: facts describing which locations are connected ----
    connected(a,b).
    connected(b,a).
    connected(b,c).
    connected(c,b).

    % A rule: the robot can move between connected locations.
    % This is the Prolog form of  Connected(X,Y) -> CanMove(X,Y).
    can_move(X,Y) :- connected(X,Y).

    % ---- Task 7: the verifier used to check a proposed plan step ----
    valid_move(X,Y) :- connected(X,Y).

    % Reachability, to show the difference between a one-step rule and its
    % transitive closure: valid_move(a,c) fails, but a->c IS reachable via b.
    reachable(X,Y) :- connected(X,Y).
    reachable(X,Y) :- connected(X,Z), connected(Z,Y).

    % ---- Task 8: a separate small example of chained inference ----
    wet_road.
    slippery     :- wet_road.
    reduce_speed :- slippery.
""")

pl_path = Path("planner.pl")
pl_path.write_text(PLANNER_PL)
print(f"\nWrote {pl_path} ({len(PLANNER_PL.splitlines())} lines)\n")
print(PLANNER_PL)

SWI-Prolog found at: /opt/homebrew/bin/swipl

Wrote planner.pl (24 lines)

% planner.pl -- warehouse knowledge base and an independent move verifier.

% ---- Task 6: facts describing which locations are connected ----
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

% A rule: the robot can move between connected locations.
% This is the Prolog form of  Connected(X,Y) -> CanMove(X,Y).
can_move(X,Y) :- connected(X,Y).

% ---- Task 7: the verifier used to check a proposed plan step ----
valid_move(X,Y) :- connected(X,Y).

% Reachability, to show the difference between a one-step rule and its
% transitive closure: valid_move(a,c) fails, but a->c IS reachable via b.
reachable(X,Y) :- connected(X,Y).
reachable(X,Y) :- connected(X,Z), connected(Z,Y).

% ---- Task 8: a separate small example of chained inference ----
wet_road.
slippery     :- wet_road.
reduce_speed :- slippery.



In [12]:
def prolog_query(goal, program="planner.pl"):
    """Run one query against SWI-Prolog in a fresh process and return true/false.

    Each call is an independent process: Prolog is genuinely a separate reasoner,
    not a Python function dressed up as one.
    """
    if not PROLOG:
        return None
    cmd = [PROLOG, "-q", "-g",
           f"( {goal} -> write(true) ; write(false) ), nl, halt",
           "-t", "halt", program]
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        text = out.stdout.strip().splitlines()
        return text[-1].strip() == "true" if text else None
    except Exception as exc:
        print("  query failed:", exc)
        return None


print("Task 6 queries:\n")
for goal in ["can_move(a,b)", "can_move(a,c)", "can_move(b,c)", "can_move(c,b)"]:
    print(f"  ?- {goal:16s} {prolog_query(goal)}")

print("\n(a) Why does can_move(a,b) succeed?")
print("    can_move(X,Y) :- connected(X,Y).  Prolog unifies X=a, Y=b and then")
print("    must prove connected(a,b), which is a FACT in the database. The proof")
print("    is one rule application followed by a fact lookup.")
print("\n(b) Why does can_move(a,c) NOT succeed?")
print("    connected(a,c) is not a fact, and no rule derives it. Under Prolog's")
print("    closed-world assumption, what cannot be proved is taken to be false,")
print("    so the query fails. Note this is failure-to-prove, not a proof of")
print("    falsity -- the distinction matters whenever the knowledge base is")
print("    incomplete rather than merely finite.")
print("\n(c) Relation to Connected(X,Y) -> CanMove(X,Y):")
print("    The clause  can_move(X,Y) :- connected(X,Y).  IS that implication,")
print("    written as a definite (Horn) clause with the conclusion first. Prolog")
print("    uses it in one direction only -- backward, from goal to subgoal -- so")
print("    it is an inference RULE rather than a two-way logical equivalence.")

Task 6 queries:

  ?- can_move(a,b)    True
  ?- can_move(a,c)    False
  ?- can_move(b,c)    True
  ?- can_move(c,b)    True

(a) Why does can_move(a,b) succeed?
    can_move(X,Y) :- connected(X,Y).  Prolog unifies X=a, Y=b and then
    must prove connected(a,b), which is a FACT in the database. The proof
    is one rule application followed by a fact lookup.

(b) Why does can_move(a,c) NOT succeed?
    connected(a,c) is not a fact, and no rule derives it. Under Prolog's
    closed-world assumption, what cannot be proved is taken to be false,
    so the query fails. Note this is failure-to-prove, not a proof of
    falsity -- the distinction matters whenever the knowledge base is
    incomplete rather than merely finite.

(c) Relation to Connected(X,Y) -> CanMove(X,Y):
    The clause  can_move(X,Y) :- connected(X,Y).  IS that implication,
    written as a definite (Horn) clause with the conclusion first. Prolog
    uses it in one direction only -- backward, from goal to subgoal -- so


In [13]:
print("Task 7 -- using Prolog to check the plan Python proposed.\n")

# Extract the Move steps from the plan the Python planner generated.
proposed_moves = []
for action in result.plan:
    if action.name.startswith("Move("):
        x, y = action.name[5:-1].split(",")
        proposed_moves.append((x.lower(), y.lower()))

print(f"Python's plan : {[a.name for a in result.plan]}")
print(f"Movement steps: {proposed_moves}\n")

print("| proposed step | Prolog: valid_move? | verdict |")
print("|---|:---:|---|")
all_ok = True
for x, y in proposed_moves:
    ok = prolog_query(f"valid_move({x},{y})")
    all_ok = all_ok and bool(ok)
    print(f"| Move({x},{y}) | {ok} | {'supported by the KB' if ok else 'NOT SUPPORTED'} |")

# The challenge: a step the Python planner never proposed.
print(f"| Move(a,c) *(hypothetical)* | {prolog_query('valid_move(a,c)')} | "
      f"not supported -- no direct A-C connection |")

print(f"\nEvery movement in the generated plan is independently supported: {all_ok}")
assert all_ok

print("\nChallenge -- suppose the planner had proposed Move(a,c):")
print(f"  ?- valid_move(a,c).   {prolog_query('valid_move(a,c)')}")
print(f"  ?- reachable(a,c).    {prolog_query('reachable(a,c)')}")
print("\n  These two answers together are the useful result. Move(a,c) is NOT a")
print("  legal single action -- so a plan containing it is invalid. But a IS")
print("  reachable from c in two steps, so the underlying TASK is achievable.")
print("  A verifier that only reported 'false' would conflate 'this step is")
print("  illegal' with 'this goal is impossible'. Separating the one-step rule")
print("  from its transitive closure keeps those distinct.")

Task 7 -- using Prolog to check the plan Python proposed.

Python's plan : ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)']
Movement steps: [('a', 'b'), ('b', 'c')]

| proposed step | Prolog: valid_move? | verdict |
|---|:---:|---|


| Move(a,b) | True | supported by the KB |


| Move(b,c) | True | supported by the KB |


| Move(a,c) *(hypothetical)* | False | not supported -- no direct A-C connection |

Every movement in the generated plan is independently supported: True

Challenge -- suppose the planner had proposed Move(a,c):


  ?- valid_move(a,c).   False


  ?- reachable(a,c).    True

  These two answers together are the useful result. Move(a,c) is NOT a
  legal single action -- so a plan containing it is invalid. But a IS
  reachable from c in two steps, so the underlying TASK is achievable.
  A verifier that only reported 'false' would conflate 'this step is
  illegal' with 'this goal is impossible'. Separating the one-step rule
  from its transitive closure keeps those distinct.


In [14]:
print("Task 8 -- chained inference.\n")
print(f"  ?- reduce_speed.   {prolog_query('reduce_speed')}")
print()
print("Why the query succeeds -- Prolog works backwards from the goal:")
print("  1. goal reduce_speed  matches the rule head  reduce_speed :- slippery.")
print("     so it must now prove  slippery")
print("  2. goal slippery      matches                slippery :- wet_road.")
print("     so it must now prove  wet_road")
print("  3. goal wet_road      is a FACT -- proved, and the chain unwinds.")
print()
print("As a sequence of implications:")
print()
print("    WetRoad                      (fact)")
print("    WetRoad -> Slippery          (rule)")
print("    Slippery -> ReduceSpeed      (rule)")
print("    ----------------------------------------")
print("    therefore ReduceSpeed        (conclusion, by modus ponens twice)")
print()
print("Fact => Rule => Rule => Conclusion, exactly the form the handout asks for.")
print("The programmer supplies the facts and rules; Prolog supplies the inference.")

Task 8 -- chained inference.

  ?- reduce_speed.   True

Why the query succeeds -- Prolog works backwards from the goal:
  1. goal reduce_speed  matches the rule head  reduce_speed :- slippery.
     so it must now prove  slippery
  2. goal slippery      matches                slippery :- wet_road.
     so it must now prove  wet_road
  3. goal wet_road      is a FACT -- proved, and the chain unwinds.

As a sequence of implications:

    WetRoad                      (fact)
    WetRoad -> Slippery          (rule)
    Slippery -> ReduceSpeed      (rule)
    ----------------------------------------
    therefore ReduceSpeed        (conclusion, by modus ponens twice)

Fact => Rule => Rule => Conclusion, exactly the form the handout asks for.
The programmer supplies the facts and rules; Prolog supplies the inference.


### Prolog reflection

**1. What is the difference between a fact and a rule?** A fact is an
unconditional assertion — `connected(a,b).` — a clause with an empty body, true
in every model of the program. A rule is a conditional — `can_move(X,Y) :-
connected(X,Y).` — a clause whose head holds *whenever* its body can be proved.
Facts are the ground truth; rules are the machinery that derives more from it.

**2. How does a query correspond to asking whether something follows from a
knowledge base?** A query `?- G.` asks Prolog to construct a proof of `G` from
the clauses. Success means $KB \vdash G$ — `G` is derivable. Two caveats matter.
Prolog uses SLD resolution, a *specific* proof procedure, so it is derivability
under that strategy rather than classical entailment in general; and failure
means only "no proof was found", which under the closed-world assumption is
*treated* as falsity. For a finite, complete knowledge base like this warehouse
the two coincide — but the distinction is exactly what makes `can_move(a,c)`
returning `false` a statement about the knowledge base rather than about the
world.

**3. Why use Prolog to verify a Python-generated plan?** Because the two systems
are **independent**. They share no code, no data structures, and no author's
assumptions. The Python planner could have a bug in `applicable`; Prolog would
still refuse to prove `valid_move(a,c)`, because its answer derives from its own
facts and its own resolution engine. Agreement between independent systems is
evidence; agreement between a program and its own explanation is not.

**4. What does an independent verifier add when the plan came from an LLM?** It
converts a *plausibility* judgement into a *derivability* one. An LLM's output is
optimised to look right, and Task 5 demonstrated that a fluent explanation
accompanies an invalid plan just as readily as a valid one. A verifier does not
assess plausibility at all — it either constructs a proof or it does not, and it
cannot be persuaded by confident phrasing.

This is the *generate-and-check* pattern, and its value comes from asymmetry:
generating a plan is hard and error-prone, while checking one is cheap and
mechanical. That asymmetry is what lets an unreliable generator be used safely,
provided the checker is genuinely independent — which is why it matters that
Prolog runs in a separate process with its own knowledge base, and not as a
Python function that reuses the planner's action definitions.

---

## Consolidated results

In [15]:
print("=" * 76)
print("RESULT SUMMARY".center(76))
print("=" * 76)
print(f"\n[Task 0] I = {{{', '.join(sorted(INITIAL))}}},  G = {{{', '.join(sorted(GOAL))}}}")
print(f"         {len(ACTIONS)} ground actions; applicable in I: "
      f"{[a.name for a in applicable_actions(INITIAL, ACTIONS)]}")
print(f"         PickUp(Package,A) applicable: True   Drop(Package,C) applicable: False")

print(f"\n[Task 1] Hand-built plan verified: 4 actions, goal achieved")
print(f"         The handout's illustrative sequence FAILS verification at")
print(f"         PickUp(Package,B) -- plausible but invalid")

print(f"\n[Task 2/3] BFS planner")
print(f"         Test A original      : plan of {result.length} actions, "
      f"{result.n_expanded} states expanded, verified valid")
print(f"         Test B PickUp removed: no plan (frontier exhausted)")
print(f"         Test B2 B-C removed  : no plan (frontier exhausted)")
print(f"         Test C with Idle     : still returns the {res_c.length}-action plan")
print(f"         Test C broken planner: returns a {bad.length}-action plan that")
print(f"                                FAILS independent verification")

print(f"\n[Task 4] Reachable state space: {len(states)} states (bound 3 x 4 = 12)")
print(f"         -> finite, so 'no plan found' is a proof, not a timeout")

print(f"\n[Prolog] swipl available: {bool(PROLOG)}")
print(f"         can_move(a,b) = {prolog_query('can_move(a,b)')}, "
      f"can_move(a,c) = {prolog_query('can_move(a,c)')}")
print(f"         all Move steps in the generated plan independently verified: {all_ok}")
print(f"         reduce_speed = {prolog_query('reduce_speed')} "
      f"(wet_road => slippery => reduce_speed)")
print("\n" + "=" * 76)

                               RESULT SUMMARY                               

[Task 0] I = {At(Package,A), At(Robot,A)},  G = {At(Package,C)}
         10 ground actions; applicable in I: ['Move(A,B)', 'PickUp(Package,A)']
         PickUp(Package,A) applicable: True   Drop(Package,C) applicable: False

[Task 1] Hand-built plan verified: 4 actions, goal achieved
         The handout's illustrative sequence FAILS verification at
         PickUp(Package,B) -- plausible but invalid

[Task 2/3] BFS planner
         Test A original      : plan of 4 actions, 6 states expanded, verified valid
         Test B PickUp removed: no plan (frontier exhausted)
         Test B2 B-C removed  : no plan (frontier exhausted)
         Test C with Idle     : still returns the 4-action plan
         Test C broken planner: returns a 1-action plan that
                                FAILS independent verification

[Task 4] Reachable state space: 12 states (bound 3 x 4 = 12)
         -> finite, so 'no plan found

---

## Reflection questions

### 1. Why specify preconditions and effects before asking an LLM to write the planner?

Because they are the *content* of the problem, and everything else is
scaffolding. BFS is generic — the same twenty lines plan warehouse deliveries or
solve a puzzle — so the only thing that makes this a warehouse planner is the
action schema. If I do not write that down first, I have delegated the actual
modelling decision, and the LLM will fill it with whatever a "robot delivery
domain" usually looks like.

It also makes the output checkable. Because I had written that Drop requires
$\text{At}(Robot,\ell) \wedge \text{Holding}(P)$, I could read the generated
`Action` definitions against my table and see they matched. Without that table I
would only have been able to ask whether the code looked reasonable — and Task 3
showed that a planner missing its precondition check looks perfectly reasonable
and returns a shorter plan.

### 2. An example of an error from failing to check preconditions

Exactly the one the notebook exhibits. The broken planner applies effects
unconditionally, so it can execute Drop(Package,C) while $\text{Holding}(P)$ is
false — producing $\text{At}(Package,C)$ out of nothing. The package teleports.

The details of that failure are worth keeping: the broken planner **ran without
error**, produced **no warning**, and returned a plan that was *shorter* than the
correct one — because removing the precondition filter enlarges the successor
set, so spurious shortcuts appear and BFS finds them first. A shorter plan looks
like a better plan. Nothing about the output signals that it is fiction; only
independent replay does.

### 3. Why is a plan that "looks reasonable" not necessarily valid?

Because plausibility is a property of the *sequence of action names* and validity
is a property of the *state trajectory*. The handout's own illustrative sequence
makes the point: Move(A,B), PickUp(P,B), Move(B,C), Drop(P,C) has the right
shape, the right length, and the right actions in a sensible order. It is
invalid, because after Move(A,B) the package is still at A and PickUp(P,B) fails
its precondition.

The gap is a single unstated assumption — that the package moved with the robot.
Reading the action names cannot expose it, because the names do not encode the
state. Only recomputing the states does. This generalises: any check that
operates on the *description* of a plan rather than on its *execution* can be
fooled by a plan whose description is well-formed.

### 4. What did the LLM contribute?

The dataclass boilerplate, the BFS skeleton, the plan-printing helpers, and the
`subprocess` plumbing for calling SWI-Prolog — that last one genuinely saved
time, since the incantation for running a single Prolog goal non-interactively
and getting a clean boolean back is fiddly.

What it did not contribute was the domain model, the decision to test a
deliberately broken planner, or the realisation that the goal test must be a
subset rather than an equality.

### 5. What did I have to verify independently?

All three of the corrections in Task 2, and they form a pattern worth naming.
The `state == goal` bug failed **loudly** — the planner reported no plan for a
solvable problem, so it could not be missed. The visited-set bug failed
**quietly** — correct answers, inflated expansion counts, and an approach that
would not scale. The delete-before-add bug failed **invisibly** — no warehouse
action has overlapping add and delete lists, so it changed no output at all and
would have sat there until the domain grew.

Only the first would have been caught by running the program. The second needed
someone to look at the reported numbers and ask whether they made sense; the
third needed someone to read the code against the specification. Generated code
should be audited at all three levels, because "it produces the right answer" is
the weakest of the three checks.

### 6. Where is logical reasoning being used?

In two places, doing two different jobs. **Applicability**: $S \models
\text{Pre}(a)$, implemented as `pre_pos <= state and not (pre_neg & state)` —
this is entailment over a set of ground propositions. **State update**: $S' = (S
\setminus \text{Del}) \cup \text{Add}$ — this is the logical semantics of what an
action *means*, and the delete-before-add ordering is part of that semantics, not
an implementation detail.

The Prolog extension adds a third, and a different kind: Prolog performs genuine
*derivation* by SLD resolution, proving `can_move(a,b)` from a rule and a fact,
rather than merely testing set membership. That is the step from "checking
propositions" to "inferring new ones".

### 7. How is planning related to the search algorithms from the previous module?

Planning **is** search — the same algorithm over a different state space. The
correspondence is exact:

| Search lab (grid) | This lab (planning) |
|---|---|
| state = `(row, col)` | state = a set of ground propositions |
| actions = Up/Down/Left/Right | actions = Move / PickUp / Drop |
| successor = move if not a wall | successor = apply effects if preconditions hold |
| goal test = reached cell `G` | goal test = $G \subseteq S$ |
| BFS optimal because moves cost 1 | BFS optimal because actions cost 1 |

The one real difference is where the successor function comes from. In the grid,
it is read off a map — a lookup. In planning, it is *computed by logical
inference* at every step: which actions are applicable is not stored anywhere but
derived from the current state and the action schemas.

That difference has consequences. The grid's state space is given explicitly and
can be drawn; the planner's is defined implicitly by the schemas and is
exponential in the number of propositions. Here it happens to be 12 states and
BFS enumerates it comfortably. Add ten packages and the space explodes, which is
why real planners use heuristics derived from the logical structure itself
(ignore-delete-lists relaxations, landmarks) rather than blind BFS — the same
move from blind to informed search that the Search lab made with A\*.

---

## Takeaway

$$\textbf{Understand} \to \textbf{Specify} \to \textbf{Generate} \to \textbf{Execute} \to \textbf{Verify}$$

The lab's structure enforces the order. The specification in Task 0 is what made
the generated code checkable; execution in Task 3 is what exposed the broken
planner; and independent verification — in Python by replay, in Prolog by
derivation — is what distinguishes a plan that is *valid* from one that merely
*reads* well.

---

## Reference

S. Russell and P. Norvig, *Artificial Intelligence: A Modern Approach*, 4th ed. —
logical agents, STRIPS-style planning representations, and forward state-space
search.